In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# This notebook will run the baseline model provided by the competition. It can be found in the official git repository: https://github.com/royerlab/kaggle-cell-tracking-competition/tree/main

# Clonning repository


In [ ]:
# # Clonning Git repository
# !git clone https://github.com/royerlab/kaggle-cell-tracking-competition.git
# %cd kaggle-cell-tracking-competition

In [ ]:
# !ls #Elements in the git repository

In [ ]:
# # -------------------------- Installing necessary dependencies (in case they are not installed) ----------------------------
# import importlib
# import subprocess, sys
# packages = [
#     "torch",
#     "zarr",
#     "polars",
#     "scipy",
#     "napari",
#     "tracksdata"
# ]
# for pkg in packages:
#     try:
#         module = importlib.import_module(pkg)
#         version = getattr(module, "__version__", "Unknown")
#         print(f"Already:  {pkg:<12} {version}")
#     except ImportError:
#         print(f" {pkg:<12} NOT INSTALLED")
#         print(f"Installing package: {pkg}:")
#         try:
#             subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
#             print(f"---- -- Package {pkg} successfully installed ------ --")
#         except ImportError:
#             print(f"---- --Package {pkg} was NOT installed ------ --")

# Defining input data

In [1]:
from pathlib import Path
import os

In [2]:
from pathlib import Path
import os
data_path = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
training_path =  data_path/'train'
test_path = data_path/'test'

#------------ Listing folders in train -----------
print("Input contents:", os.listdir(training_path)[:5],"\n")
print(" First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)")

Input contents: ['6bba_2540cd90.geff', '44b6_0b24845f.geff', '44b6_996155de.geff', '44b6_0c582fdc.geff', '6bba_cf35214c.zarr'] 

 First 5 elements in train folder, it contains '.geff' folders (annotated graphs) and '.zarr' folder (videos)


# Making a prediction with the baseline model

The notebook used to run the baseline can be found here: https://www.kaggle.com/code/thibautgoldsborough/unet-baseline-inference-submission

## Configuration

In [3]:

COMP_DIR = data_path
TEST_DIR = test_path

ARTIFACTS = "/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"

REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"

# --- Test-time options (tune these) ---------------------------------------
# Model checkpoint (relative to the repo, or an absolute path to your own).
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

# GT: Ground-Truth
# Detection peak threshold. GT is sparse so the detector is poorly calibrated;
# 0.5 is too low, ~0.99 scored best in a sweep.
DET_THRESHOLD = 0.99

UNET_BATCH_SIZE = 4          # frame-pairs per UNet forward; lower if you OOM
SLICE = ":5"                   # e.g. ":5" to predict only the first 5 videos; "" = all

# Linking. ILP = global, flow-consistent (cleaner tracks; ~0.73->0.79).
# Set False for the faster greedy linker.
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0           # weight on edge probability
ILP_APPEARANCE_WEIGHT = 0.1      # cost of a track appearing
ILP_DISAPPEARANCE_WEIGHT = 0.1   # cost of a track disappearing
ILP_DIVISION_WEIGHT = 1.0        # cost of a division; lower (~0.2) allows more splits
# --------------------------------------------------------------------------

## Offline install & setup
Install the dependencies from the bundled wheels (no internet), then copy the repo source and weights into a writable location and put the package on the import path

In [5]:
import glob
import os
import subprocess
import sys
import shutil

# 1. Find all wheels, but filter OUT numpy wheels to avoid breaking C-extensions
wheels = [
    f for f in glob.glob(f"{ARTIFACTS}/wheels/*.whl")
    if "numpy" not in os.path.basename(f).lower()
]

# 2. Install only the required non-NumPy wheels without upgrading dependencies
subprocess.run(
    [
        "pip", "install", 
        "--no-index", 
        "--find-links", f"{ARTIFACTS}/wheels",
        "--no-deps",
        *wheels
    ],
    check=True,
)

# 3. Copy repo and weights as usual
shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/src")

Looking in links: /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/imagecodecs-2026.6.26-cp312-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/shellingham-1.5.4-py2.py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/numcodecs-0.15.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/bidict-0.23.1-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts/wheels/requests-2.34.2-py3-none-any.whl
Processing /kaggle/input/datasets/thibautgoldsborough/cellmot-bas

In [ ]:
# import sys
# import shutil
# import subprocess

# subprocess.run(
#     ["pip", "install", "--no-index", "--find-links", f"{ARTIFACTS}/wheels", "--no-deps", "donfig",
#      "tracksdata", "zarr>=3.0.10", "pyscipopt"],
#     check=True)

# shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
# shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
# sys.path.insert(0, f"{REPO_DIR}/src")

# print("Weights:", os.listdir(f"{REPO_DIR}/weights/{METHOD}/split_0"))

## Inference on all test videos
Build a one-fold splits file listing every test video, then run prediction. PYTHONPATH=src makes the tracking_cellmot package importable without an internet install. Each video's tracks are exported as a .geff graph.

In [6]:
import json

test_stems = sorted(f[:-5] for f in os.listdir(TEST_DIR) if f.endswith(".zarr"))
print(f"{len(test_stems)} test videos")
print(f"One element in test_stems is {test_stems[0]}")

with open(f"{REPO_DIR}/kaggle_test_splits.json", "w") as f:
    json.dump([{"split": 0, "train": [], "test": test_stems}], f)

4 test videos
One element in test_stems is 44b6_0113de3b


In [7]:
#---------- Reading back the json file just created ------------
with open(f"{REPO_DIR}/kaggle_test_splits.json", "r") as f:
    json_file = json.load(f)

json_file

[{'split': 0,
  'train': [],
  'test': ['44b6_0113de3b',
   '44b6_0b24845f',
   '6bba_05b6850b',
   '6bba_05db0fb1']}]

In [8]:
import subprocess

# ----------------- command to run "predict_unet_transformer.py" which predicts data ------------------------
cmd = [
    "python", "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR), "--splits", "kaggle_test_splits.json", "--split", "0",
    "--weights", str(WEIGHTS), "--unet-batch-size", str(UNET_BATCH_SIZE),
    "--det-threshold", str(DET_THRESHOLD),
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")
if SLICE:
    cmd += ["--slice", SLICE]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)

python scripts/predict_unet_transformer.py --data-dir /kaggle/input/competitions/biohub-cell-tracking-during-development/test --splits kaggle_test_splits.json --split 0 --weights weights/unet_transformer/split_0/edge_predictor_best.pth --unet-batch-size 1 --det-threshold 0.99 --ilp-edge-weight -1.0 --ilp-appearance-weight 0.1 --ilp-disappearance-weight 0.1 --ilp-division-weight 1.0 --use-ilp --slice :1


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


Fold 0: 1 datasets | weights=weights/unet_transformer/split_0/edge_predictor_best.pth | device=cuda | window_size=2 | pool_kernel_um=3.0
Saved 1 predictions to /kaggle/working/repo/predictions/unknown/unet_transformer/split_0


CompletedProcess(args=['python', 'scripts/predict_unet_transformer.py', '--data-dir', '/kaggle/input/competitions/biohub-cell-tracking-during-development/test', '--splits', 'kaggle_test_splits.json', '--split', '0', '--weights', 'weights/unet_transformer/split_0/edge_predictor_best.pth', '--unet-batch-size', '1', '--det-threshold', '0.99', '--ilp-edge-weight', '-1.0', '--ilp-appearance-weight', '0.1', '--ilp-disappearance-weight', '0.1', '--ilp-division-weight', '1.0', '--use-ilp', '--slice', ':1'], returncode=0)

# Build submission.csv
Flatten the predicted .geff graphs into the competition's CSV format: one node row per detection (t, z, y, x) and one edge row per link (source_id, target_id)

In [9]:
from pathlib import Path

import pandas as pd
import tracksdata as td

geffs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"{len(geffs)} prediction graphs")

rows = []
for g in geffs:
    name = g.stem
    graph = td.graph.IndexedRXGraph.from_geff(g)
    graph = graph[0] if isinstance(graph, tuple) else graph
    for r in graph.node_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "node", "node_id": int(r["node_id"]),
            "t": int(r["t"]), "z": int(round(r["z"])), "y": int(round(r["y"])),
            "x": int(round(r["x"])), "source_id": -1, "target_id": -1,
        })
    for r in graph.edge_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(r["source_id"]), "target_id": int(r["target_id"]),
        })

submission = pd.DataFrame(
    rows,
    columns=["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"],
)
submission.index.name = "id"
submission.to_csv("submission_baseline.csv")
print(f"Wrote submission.csv with {len(submission)} rows")
submission.head()

/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


1 prediction graphs
Wrote submission.csv with 49533 rows


,dataset,row_type,node_id,t,z,y,x,source_id,target_id
id,,,,,,,,,
0,44b6_0113de3b,node,0,0,1,8,52,-1,-1
1,44b6_0113de3b,node,1,0,1,8,72,-1,-1
2,44b6_0113de3b,node,2,0,1,24,64,-1,-1
3,44b6_0113de3b,node,3,0,1,64,60,-1,-1
4,44b6_0113de3b,node,4,0,1,100,36,-1,-1
